In [6]:
import json
import numpy as np
from rliable import library as rly
from rliable import metrics
from pathlib import Path
from loguru import logger as console_logger

def get_scores(
    data: dict,
    m: int,
    tasks: list[str],
    perturbation_tests: list[str],
    difficulties: list[str],
    metric_name: str,
    nb_seeds: int = 5,
) -> np.ndarray:
    n = nb_seeds
    scores = np.zeros((m, n), dtype=np.float32)
    m_index = 0

    for task_key, perturbation_tests_val in data.items():
        if task_key in tasks:
            for perturbation_test_key, difficulty_val in perturbation_tests_val.items():
                if perturbation_test_key in perturbation_tests:
                    for difficulty in difficulties:
                        scores[m_index] = np.array(
                            difficulty_val[difficulty][metric_name]
                        )
                        m_index += 1

    return scores


def get_no_perturb_scores(
    data: dict,
    m: int,
    tasks: list[str],
    metric_name: str,
    nb_seeds: int = 5
) -> np.ndarray:
    n = nb_seeds
    scores = np.zeros((m, n), dtype=np.float32)
    m_index = 0

    for task in tasks:
        task_scores = data[task]["no_perturbation_test"][metric_name]
        scores[m_index] = np.array(task_scores)
        m_index += 1

    return scores


def recursive_override(src: dict, dest: dict):
    for key, src_val in src.items():
        dest_val = dest[key]
        if isinstance(src_val, dict) and isinstance(dest_val, dict):
            recursive_override(src_val, dest_val)
        else:
            dest[key] = src_val

# Assemble

In [7]:
algo_name = "samg"
filenames = [f"results/test/partial/samg_test_results-{i}.json" for i in range(1, 9)]

metrics_data = {algo_name: {}}

print("Loading and assembling files...")

for fname in filenames:
    file_path = Path(fname)
    assert file_path.exists()
    
    with open(file_path, 'r') as f:
        data = json.load(f)
        
        metrics_data[algo_name].update(data[algo_name])
        print(f"Loaded tasks from {fname}: {list(data[algo_name].keys())}")

# Extract standard params from the loaded data
sample_task = list(metrics_data[algo_name].keys())[0]
sample_metric = metrics_data[algo_name][sample_task]['no_perturbation_test']['return']
nb_seeds = len(sample_metric)

print("-" * 30)
print(f"Assembly Complete.")
print(f"Total Tasks: {len(metrics_data[algo_name])}")
print(f"Seeds detected: {nb_seeds}")

Loading and assembling files...
Loaded tasks from results/test/partial/samg_test_results-1.json: ['pushcubetest-v1']
Loaded tasks from results/test/partial/samg_test_results-2.json: ['pullcubetest-v1']
Loaded tasks from results/test/partial/samg_test_results-3.json: ['pickcubetest-v1']
Loaded tasks from results/test/partial/samg_test_results-4.json: ['pokecubetest-v1']
Loaded tasks from results/test/partial/samg_test_results-5.json: ['pullcubetooltest-v1']
Loaded tasks from results/test/partial/samg_test_results-6.json: ['liftpeguprighttest-v1']
Loaded tasks from results/test/partial/samg_test_results-7.json: ['unitreeg1placeappleinbowltest-v1']
Loaded tasks from results/test/partial/samg_test_results-8.json: ['unitreeg1transportboxtest-v1']
------------------------------
Assembly Complete.
Total Tasks: 8
Seeds detected: 5


# Aggregate

In [ ]:
console_logger.info("Aggregating scores...")

camera_perturbation_tests = ['camera_pose_test', 'camera_fov_test']
lighting_perturbation_tests = [
    'lighting_direction_test', 'lighting_color_test']
color_perturbation_tests = [
    'mo_color_test', 'ro_color_test', 'table_color_test', 'ground_color_test']
texture_perturbation_tests = [
    'mo_texture_test', 'ro_texture_test', 'table_texture_test', 'ground_texture_test']
all_perturbation_tests = camera_perturbation_tests + lighting_perturbation_tests + \
    color_perturbation_tests + texture_perturbation_tests

assert len(all_perturbation_tests) == 12

difficulties = ["easy","medium","hard"]

ro_tests = ['ro_color_test', 'ro_texture_test']

task_perturbation_tests = {
    'pushcubetest-v1': all_perturbation_tests,
    'pullcubetest-v1': all_perturbation_tests,
    # This env doesn't have RO
    'pickcubetest-v1': set(all_perturbation_tests) - set(ro_tests),
    'pokecubetest-v1': all_perturbation_tests,
    'pullcubetooltest-v1': all_perturbation_tests,
    # This env doesn't have RO
    'liftpeguprighttest-v1': set(all_perturbation_tests) - set(ro_tests),
    'unitreeg1placeappleinbowltest-v1': all_perturbation_tests,
    # This env doesn't have RO
    'unitreeg1transportboxtest-v1': set(all_perturbation_tests) - set(ro_tests),
}

task_ids = [
    "pushcubetest-v1",
    "pullcubetest-v1",
    "pickcubetest-v1",
    "pokecubetest-v1",
    "pullcubetooltest-v1",
    "liftpeguprighttest-v1",
    "unitreeg1placeappleinbowltest-v1",
    "unitreeg1transportboxtest-v1"
]
reps = 50_000
confidence_interval_size = 0.95
random_state = np.random.default_rng(seed=12345)
algo_aggregated_scores = {
    algo_name: {
        "no_perturb": {

        },
        "perturb_per_diff_per_cat": {

        },
        "perturb_indiv": {

        }
    }
}
m = len(task_ids)
console_logger.info("Computing overall-no_perturbation_test scores...")
no_perturb_overall_return_scores = get_no_perturb_scores(
    data=metrics_data[algo_name],
    m=m,
    tasks=task_ids,
    nb_seeds=nb_seeds,
    metric_name="return"
)
no_perturb_overall_success_at_end_scores = get_no_perturb_scores(
    data=metrics_data[algo_name],
    m=m,
    tasks=task_ids,
    nb_seeds=nb_seeds,
    metric_name="success_at_end"
)
agg_no_perturb_overall_return_scores, agg_no_perturb_overall_return_score_cis = rly.get_interval_estimates(
    {algo_name: no_perturb_overall_return_scores},
    lambda x: np.array([metrics.aggregate_iqm(x)]),
    reps=reps,
    confidence_interval_size=confidence_interval_size,
    random_state=random_state
)
agg_no_perturb_overall_success_at_end_scores, agg_no_perturb_overall_success_at_end_score_cis = rly.get_interval_estimates(
    {algo_name: no_perturb_overall_success_at_end_scores},
    lambda x: np.array([metrics.aggregate_iqm(x)]),
    reps=reps,
    confidence_interval_size=confidence_interval_size,
    random_state=random_state
)
algo_aggregated_scores[algo_name]["no_perturb"]["overall"] = {
    "return": {
        'iqm': agg_no_perturb_overall_return_scores[algo_name].tolist(),
        'ci': agg_no_perturb_overall_return_score_cis[algo_name].tolist()
    },
    "success_at_end": {
        'iqm': agg_no_perturb_overall_success_at_end_scores[algo_name].tolist(),
        'ci': agg_no_perturb_overall_success_at_end_score_cis[algo_name].tolist()
    }
}
for difficulty in difficulties:
    algo_aggregated_scores[algo_name]["perturb_indiv"][difficulty] = {}
    difficulty_list = [difficulty]
    camera_m = 0
    lighting_m = 0
    color_m = 0
    texture_m = 0
    for task_id in task_ids:
        task_perturbations = set(task_perturbation_tests[task_id])
        camera_m += len(list(task_perturbations &
                        set(camera_perturbation_tests)))
        lighting_m += len(list(task_perturbations &
                            set(lighting_perturbation_tests)))
        color_m += len(list(task_perturbations &
                        set(color_perturbation_tests)))
        texture_m += len(list(task_perturbations &
                            set(texture_perturbation_tests)))

    overall_m = camera_m + lighting_m + color_m + texture_m

    console_logger.info(f"Difficulty: {difficulty}")
    console_logger.info(f"overall_m: {overall_m}")
    console_logger.info(f"camera_m: {camera_m}")
    console_logger.info(f"lighting_m: {lighting_m}")
    console_logger.info(f"color_m: {color_m}")
    console_logger.info(f"texture_m: {texture_m}")

    console_logger.info("Computing Overall scores...")
    overall_return_scores = get_scores(
        metrics_data[algo_name],
        m=overall_m,
        tasks=task_ids,
        perturbation_tests=all_perturbation_tests,
        difficulties=difficulty_list,
        nb_seeds=nb_seeds,
        metric_name="return"
    )
    overall_agg_return_scores, overall_agg_return_score_cis = rly.get_interval_estimates(
        {algo_name: overall_return_scores},
        lambda x: np.array([metrics.aggregate_iqm(x)]),
        reps=reps,
        confidence_interval_size=confidence_interval_size,
        random_state=random_state
    )
    overall_success_at_end_scores = get_scores(
        metrics_data[algo_name],
        m=overall_m,
        tasks=task_ids,
        perturbation_tests=all_perturbation_tests,
        difficulties=difficulty_list,
        nb_seeds=nb_seeds,
        metric_name="success_at_end"
    )
    overall_agg_success_at_end_scores, overall_agg_success_at_end_score_cis = rly.get_interval_estimates(
        {algo_name: overall_success_at_end_scores},
        lambda x: np.array([metrics.aggregate_iqm(x)]),
        reps=reps,
        confidence_interval_size=confidence_interval_size,
        random_state=random_state
    )
    console_logger.info("Computing Camera scores...")
    camera_return_scores = get_scores(
        metrics_data[algo_name],
        m=camera_m,
        tasks=task_ids,
        perturbation_tests=camera_perturbation_tests,
        difficulties=difficulty_list,
        nb_seeds=nb_seeds,
        metric_name="return"
    )
    camera_success_at_end_scores = get_scores(
        metrics_data[algo_name],
        m=camera_m,
        tasks=task_ids,
        perturbation_tests=camera_perturbation_tests,
        difficulties=difficulty_list,
        nb_seeds=nb_seeds,
        metric_name="success_at_end"
    )
    camera_agg_return_scores, camera_agg_return_score_cis = rly.get_interval_estimates(
        {algo_name: camera_return_scores},
        lambda x: np.array([metrics.aggregate_iqm(x)]),
        reps=reps,
        confidence_interval_size=confidence_interval_size,
        random_state=random_state
    )
    camera_agg_success_at_end_scores, camera_agg_success_at_end_score_cis = rly.get_interval_estimates(
        {algo_name: camera_success_at_end_scores},
        lambda x: np.array([metrics.aggregate_iqm(x)]),
        reps=reps,
        confidence_interval_size=confidence_interval_size,
        random_state=random_state
    )
    console_logger.info("Computing Lighting scores...")
    lighting_return_scores = get_scores(
        metrics_data[algo_name],
        m=lighting_m,
        tasks=task_ids,
        perturbation_tests=lighting_perturbation_tests,
        difficulties=difficulty_list,
        nb_seeds=nb_seeds,
        metric_name="return"
    )
    lighting_success_at_end_scores = get_scores(
        metrics_data[algo_name],
        m=lighting_m,
        tasks=task_ids,
        perturbation_tests=lighting_perturbation_tests,
        difficulties=difficulty_list,
        nb_seeds=nb_seeds,
        metric_name="success_at_end"
    )
    lighting_agg_return_scores, lighting_agg_return_score_cis = rly.get_interval_estimates(
        {algo_name: lighting_return_scores},
        lambda x: np.array([metrics.aggregate_iqm(x)]),
        reps=reps,
        confidence_interval_size=confidence_interval_size,
        random_state=random_state
    )
    lighting_agg_success_at_end_scores, lighting_agg_success_at_end_score_cis = rly.get_interval_estimates(
        {algo_name: lighting_success_at_end_scores},
        lambda x: np.array([metrics.aggregate_iqm(x)]),
        reps=reps,
        confidence_interval_size=confidence_interval_size,
        random_state=random_state
    )
    console_logger.info("Computing Color scores...")
    color_return_scores = get_scores(
        metrics_data[algo_name],
        m=color_m,
        tasks=task_ids,
        perturbation_tests=color_perturbation_tests,
        difficulties=difficulty_list,
        nb_seeds=nb_seeds,
        metric_name="return"
    )
    color_success_at_end_scores = get_scores(
        metrics_data[algo_name],
        m=color_m,
        tasks=task_ids,
        perturbation_tests=color_perturbation_tests,
        difficulties=difficulty_list,
        nb_seeds=nb_seeds,
        metric_name="success_at_end"
    )
    color_agg_return_scores, color_agg_return_score_cis = rly.get_interval_estimates(
        {algo_name: color_return_scores},
        lambda x: np.array([metrics.aggregate_iqm(x)]),
        reps=reps,
        confidence_interval_size=confidence_interval_size,
        random_state=random_state
    )
    color_agg_success_at_end_scores, color_agg_success_at_end_score_cis = rly.get_interval_estimates(
        {algo_name: color_success_at_end_scores},
        lambda x: np.array([metrics.aggregate_iqm(x)]),
        reps=reps,
        confidence_interval_size=confidence_interval_size,
        random_state=random_state
    )
    console_logger.info("Computing Texture scores...")
    texture_return_scores = get_scores(
        metrics_data[algo_name],
        m=texture_m,
        tasks=task_ids,
        perturbation_tests=texture_perturbation_tests,
        difficulties=difficulty_list,
        nb_seeds=nb_seeds,
        metric_name="return"
    )
    texture_success_at_end_scores = get_scores(
        metrics_data[algo_name],
        m=texture_m,
        tasks=task_ids,
        perturbation_tests=texture_perturbation_tests,
        difficulties=difficulty_list,
        nb_seeds=nb_seeds,
        metric_name="success_at_end"
    )
    texture_agg_return_scores, texture_agg_return_score_cis = rly.get_interval_estimates(
        {algo_name: texture_return_scores},
        lambda x: np.array([metrics.aggregate_iqm(x)]),
        reps=reps,
        confidence_interval_size=confidence_interval_size,
        random_state=random_state
    )
    texture_agg_success_at_end_scores, texture_agg_success_at_end_score_cis = rly.get_interval_estimates(
        {algo_name: texture_success_at_end_scores},
        lambda x: np.array([metrics.aggregate_iqm(x)]),
        reps=reps,
        confidence_interval_size=confidence_interval_size,
        random_state=random_state
    )
    algo_aggregated_scores[algo_name]["perturb_per_diff_per_cat"][difficulty] = {
        'overall': {
            "return": {
                'iqm': overall_agg_return_scores[algo_name].tolist(),
                'ci': overall_agg_return_score_cis[algo_name].tolist()
            },
            "success_at_end": {
                'iqm': overall_agg_success_at_end_scores[algo_name].tolist(),
                'ci': overall_agg_success_at_end_score_cis[algo_name].tolist()
            }
        },
        'camera': {
            "return": {
                'iqm': camera_agg_return_scores[algo_name].tolist(),
                'ci': camera_agg_return_score_cis[algo_name].tolist()
            },
            "success_at_end": {
                'iqm': camera_agg_success_at_end_scores[algo_name].tolist(),
                'ci': camera_agg_success_at_end_score_cis[algo_name].tolist()
            }
        },
        'lighting': {
            "return": {
                'iqm': lighting_agg_return_scores[algo_name].tolist(),
                'ci': lighting_agg_return_score_cis[algo_name].tolist()
            },
            "success_at_end": {
                'iqm': lighting_agg_success_at_end_scores[algo_name].tolist(),
                'ci': lighting_agg_success_at_end_score_cis[algo_name].tolist()
            }
        },
        'color': {
            "return": {
                'iqm': color_agg_return_scores[algo_name].tolist(),
                'ci': color_agg_return_score_cis[algo_name].tolist()
            },
            "success_at_end": {
                'iqm': color_agg_success_at_end_scores[algo_name].tolist(),
                'ci': color_agg_success_at_end_score_cis[algo_name].tolist()
            }
        },
        "texture": {
            "return": {
                'iqm': texture_agg_return_scores[algo_name].tolist(),
                'ci': texture_agg_return_score_cis[algo_name].tolist()
            },
            "success_at_end": {
                'iqm': texture_agg_success_at_end_scores[algo_name].tolist(),
                'ci': texture_agg_success_at_end_score_cis[algo_name].tolist()
            }
        }
    }

    for task_id in task_ids:
        algo_aggregated_scores[algo_name]["perturb_indiv"][difficulty][task_id] = {
        }

        m = 1
        console_logger.info(
            f"Computing {task_id}-no_perturbation_test scores...")
        no_perturb_indiv_return_scores = get_no_perturb_scores(
            data=metrics_data[algo_name],
            m=m,
            tasks=[task_id],
            nb_seeds=nb_seeds,
            metric_name="return"
        )
        no_perturb_indiv_success_at_end_scores = get_no_perturb_scores(
            data=metrics_data[algo_name],
            m=m,
            tasks=[task_id],
            nb_seeds=nb_seeds,
            metric_name="success_at_end"
        )

        agg_no_perturb_indiv_return_scores, agg_no_perturb_indiv_return_score_cis = rly.get_interval_estimates(
            {algo_name: no_perturb_indiv_return_scores.T},
            lambda x: np.array([metrics.aggregate_iqm(x)]),
            reps=reps,
            confidence_interval_size=confidence_interval_size,
            random_state=random_state
        )
        agg_no_perturb_indiv_success_at_end_scores, agg_no_perturb_indiv_success_at_end_score_cis = rly.get_interval_estimates(
            {algo_name: no_perturb_indiv_success_at_end_scores.T},
            lambda x: np.array([metrics.aggregate_iqm(x)]),
            reps=reps,
            confidence_interval_size=confidence_interval_size,
            random_state=random_state
        )
        algo_aggregated_scores[algo_name]["no_perturb"][task_id] = {
            "return": {
                'iqm': agg_no_perturb_indiv_return_scores[algo_name].tolist(),
                'ci': agg_no_perturb_indiv_return_score_cis[algo_name].tolist()
            },
            "success_at_end": {
                'iqm': agg_no_perturb_indiv_success_at_end_scores[algo_name].tolist(),
                'ci': agg_no_perturb_indiv_success_at_end_score_cis[algo_name].tolist()
            }
        }

        for perturbation_test in task_perturbation_tests[task_id]:
            m = 1
            console_logger.info(
                f"Computing {task_id}-{perturbation_test}-{difficulty} scores...")
            perturb_indiv_return_scores = get_scores(
                metrics_data[algo_name],
                m=m,
                tasks=[task_id],
                perturbation_tests=[perturbation_test],
                difficulties=difficulty_list,
                nb_seeds=nb_seeds,
                metric_name="return"
            )
            perturb_indiv_success_at_end_scores = get_scores(
                metrics_data[algo_name],
                m=m,
                tasks=[task_id],
                perturbation_tests=[perturbation_test],
                difficulties=difficulty_list,
                nb_seeds=nb_seeds,
                metric_name="success_at_end"
            )

            agg_perturb_indiv_return_scores, agg_perturb_indiv_return_score_cis = rly.get_interval_estimates(
                {algo_name: perturb_indiv_return_scores.T},
                lambda x: np.array([metrics.aggregate_iqm(x)]),
                reps=reps,
                confidence_interval_size=confidence_interval_size,
                random_state=random_state
            )
            agg_perturb_indiv_success_at_end_scores, agg_perturb_indiv_success_at_end_score_cis = rly.get_interval_estimates(
                {algo_name: perturb_indiv_success_at_end_scores.T},
                lambda x: np.array([metrics.aggregate_iqm(x)]),
                reps=reps,
                confidence_interval_size=confidence_interval_size,
                random_state=random_state
            )

            algo_aggregated_scores[algo_name]["perturb_indiv"][difficulty][task_id][perturbation_test] = {
                "return": {
                    'iqm': agg_perturb_indiv_return_scores[algo_name].tolist(),
                    'ci': agg_perturb_indiv_return_score_cis[algo_name].tolist()
                },
                "success_at_end": {
                    'iqm': agg_perturb_indiv_success_at_end_scores[algo_name].tolist(),
                    'ci': agg_perturb_indiv_success_at_end_score_cis[algo_name].tolist()
                }
            }

console_logger.info("Saving aggregated scores...")
algo_aggregated_scores_path = Path(f"results/test/{algo_name}_final_aggregated_scores-0.json")
with open(algo_aggregated_scores_path, 'w') as f:
    json.dump(algo_aggregated_scores, f)

2025-12-02 03:45:56.021 | INFO     | __main__:<module>:1 - Aggregating scores...
2025-12-02 03:45:56.023 | INFO     | __main__:<module>:60 - Computing overall-no_perturbation_test scores...
/home/mila/b/browna/.local/lib/python3.10/site-packages/arch/bootstrap/base.py:402: FutureWarning: random_state is deprecated and will be removed in a future version. The default random number generator is changing to a NumPy Generator. To continue using RandomState, please directly pass a RandomState instance using the ``generator`` keyword argument.
  warnings.warn(
2025-12-02 03:46:00.048 | INFO     | __main__:<module>:119 - Difficulty: easy
2025-12-02 03:46:00.049 | INFO     | __main__:<module>:120 - overall_m: 90
2025-12-02 03:46:00.050 | INFO     | __main__:<module>:121 - camera_m: 16
2025-12-02 03:46:00.050 | INFO     | __main__:<module>:122 - lighting_m: 16
2025-12-02 03:46:00.050 | INFO     | __main__:<module>:123 - color_m: 29
2025-12-02 03:46:00.051 | INFO     | __main__:<module>:124 - te